# Feature Detection & Object Detection with OpenCV

**Plan**
1. Quick recap: video, mouse, trackbars, color tracking, thresholding, filters, contours
2. Feature detection: Harris, Shi-Tomasi, ORB, feature matching
3. Object detection: template matching, motion detection, Haar cascades

> Cells that use a **webcam / window** open a separate OpenCV window. Press **`q`** or **`Esc`** to close it.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

DATA = 'data/'   # change to your data folder

def show(*imgs, titles=None):
    """Show one or more images side by side (BGR or gray)."""
    plt.figure(figsize=(5 * len(imgs), 4))
    for i, img in enumerate(imgs):
        plt.subplot(1, len(imgs), i + 1)
        if img.ndim == 2:
            plt.imshow(img, cmap='gray')
        else:
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        if titles:
            plt.title(titles[i])
        plt.axis('off')
    plt.show()

---
# Part 1 — Quick Recap

## 1.1 Reading a video
A video is just a loop of images (frames). `cap.read()` returns `ret` (True/False) and the `frame`.
Use `0` instead of a file path to open the webcam.

In [ ]:
cap = cv.VideoCapture(DATA + 'vtest.avi')   # or cv.VideoCapture(0) for webcam

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:                       # end of video
        break
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)
    cv.imshow('frame', gray)
    if cv.waitKey(1) == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

## 1.2 Saving a video
`VideoWriter` needs: file name, codec (`fourcc`), FPS and frame size.

In [ ]:
cap = cv.VideoCapture(0)
w, h = int(cap.get(cv.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
out = cv.VideoWriter('output.avi', cv.VideoWriter_fourcc(*'XVID'), 20.0, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv.flip(frame, 0)         # flip upside-down
    out.write(frame)
    cv.imshow('frame', frame)
    if cv.waitKey(1) == ord('q'):
        break

cap.release()
out.release()
cv.destroyAllWindows()

## 1.3 Mouse events
We attach a **callback function** to a window. OpenCV calls it on every mouse action.

In [ ]:
print([e for e in dir(cv) if 'EVENT' in e])   # all available mouse events

In [ ]:
img = np.zeros((512, 512, 3), np.uint8)

def draw_circle(event, x, y, flags, param):
    if event == cv.EVENT_LBUTTONDBLCLK:          # double click -> draw a circle
        cv.circle(img, (x, y), 50, (255, 0, 0), -1)

cv.namedWindow('image')
cv.setMouseCallback('image', draw_circle)

while True:
    cv.imshow('image', img)
    if cv.waitKey(20) & 0xFF == 27:               # Esc  (& 0xFF keeps the last 8 bits)
        break
cv.destroyAllWindows()

## 1.4 Trackbar as a color palette
A trackbar is a slider. We read its value with `getTrackbarPos`.

In [ ]:
img = np.zeros((300, 512, 3), np.uint8)
cv.namedWindow('image')
for c in ['R', 'G', 'B']:
    cv.createTrackbar(c, 'image', 0, 255, lambda x: None)

while True:
    cv.imshow('image', img)
    if cv.waitKey(1) & 0xFF == 27:
        break
    r, g, b = [cv.getTrackbarPos(c, 'image') for c in ['R', 'G', 'B']]
    img[:] = [b, g, r]                            # OpenCV uses BGR order

cv.destroyAllWindows()

## 1.5 Measuring performance
`getTickCount()` counts clock ticks; divide by `getTickFrequency()` to get seconds.

In [ ]:
img = cv.imread(DATA + 'messi5.jpg')

e1 = cv.getTickCount()
for _ in range(10):
    img_blur = cv.medianBlur(img, 9)
e2 = cv.getTickCount()

print(f'Time: {(e2 - e1) / cv.getTickFrequency():.4f} s')

## 1.6 Color tracking (HSV)
HSV separates **color (Hue)** from **brightness**, so picking a color range is much easier than in BGR.

Steps: BGR ➜ HSV ➜ `inRange` gives a mask ➜ `bitwise_and` keeps only that color.

Use the sliders to find the range of any object.

In [ ]:
cap = cv.VideoCapture(0)
cv.namedWindow('tracking')
for name, start, end in [('LH', 0, 179), ('LS', 0, 255), ('LV', 0, 255),
                         ('UH', 179, 179), ('US', 255, 255), ('UV', 255, 255)]:
    cv.createTrackbar(name, 'tracking', start, end, lambda x: None)

while True:
    _, frame = cap.read()
    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

    lower = np.array([cv.getTrackbarPos(n, 'tracking') for n in ['LH', 'LS', 'LV']])
    upper = np.array([cv.getTrackbarPos(n, 'tracking') for n in ['UH', 'US', 'UV']])

    mask = cv.inRange(hsv, lower, upper)
    result = cv.bitwise_and(frame, frame, mask=mask)

    cv.imshow('tracking', result)
    cv.imshow('mask', mask)
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

## 1.7 Thresholding
Turn a gray image into black & white: pixel > threshold ➜ one value, otherwise ➜ another.

| Type | Pixel > 127 | Pixel ≤ 127 |
|---|---|---|
| `BINARY` | 255 | 0 |
| `BINARY_INV` | 0 | 255 |
| `TRUNC` | 127 | unchanged |
| `TOZERO` | unchanged | 0 |
| `TOZERO_INV` | 0 | unchanged |

In [ ]:
gray = cv.imread(DATA + 'messi5.jpg', cv.IMREAD_GRAYSCALE)

types = ['THRESH_BINARY', 'THRESH_BINARY_INV', 'THRESH_TRUNC', 'THRESH_TOZERO', 'THRESH_TOZERO_INV']
results = [cv.threshold(gray, 127, 255, getattr(cv, t))[1] for t in types]

show(gray, *results[:2], titles=['Original', 'BINARY', 'BINARY_INV'])
show(*results[2:], titles=['TRUNC', 'TOZERO', 'TOZERO_INV'])

### Global vs Otsu
- **Global**: *we* choose the threshold (127).
- **Otsu**: OpenCV *finds the best threshold automatically* from the histogram (we pass 0). The chosen value is returned in `ret`.

In [ ]:
ret1, th1 = cv.threshold(gray, 127, 255, cv.THRESH_BINARY)
ret2, th2 = cv.threshold(gray, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)

print('Global threshold:', ret1, '| Otsu threshold:', ret2)
show(th1, th2, titles=['Global (127)', f'Otsu ({ret2:.0f})'])

## 1.8 Filtering
- `filter2D`: apply **any kernel** you design (here: average of 5x5 pixels).
- `bilateralFilter`: removes noise but **keeps edges sharp**.

In [ ]:
img = cv.imread(DATA + 'messi5.jpg')

kernel = np.ones((5, 5), np.float32) / 25
avg = cv.filter2D(img, -1, kernel)
bilateral = cv.bilateralFilter(img, 9, 75, 75)

show(img, avg, bilateral, titles=['Original', 'filter2D (average)', 'Bilateral'])

## 1.9 Contours
A contour is a curve joining all boundary points of a shape.

| Function | What it does |
|---|---|
| `findContours` | finds the shapes (needs a binary image) |
| `drawContours` | draws them |
| `contourArea` | area of a shape |
| `arcLength` | perimeter of a shape |
| `approxPolyDP` | simplifies the shape to fewer points ➜ count corners |

In [ ]:
# a simple image with shapes
img = np.zeros((300, 600, 3), np.uint8)
cv.rectangle(img, (30, 50), (180, 250), (255, 255, 255), -1)
cv.circle(img, (300, 150), 80, (255, 255, 255), -1)
cv.fillPoly(img, [np.array([[500, 50], [420, 250], [580, 250]])], (255, 255, 255))

gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
_, thresh = cv.threshold(gray, 127, 255, cv.THRESH_BINARY)
contours, _ = cv.findContours(thresh, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

for cnt in contours:
    area = cv.contourArea(cnt)
    perimeter = cv.arcLength(cnt, True)
    approx = cv.approxPolyDP(cnt, 0.02 * perimeter, True)   # 2% of the perimeter
    corners = len(approx)
    name = {3: 'Triangle', 4: 'Rectangle'}.get(corners, 'Circle')

    cv.drawContours(img, [cnt], -1, (0, 255, 0), 3)
    x, y = approx[0][0]
    cv.putText(img, name, (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    print(f'{name:10} corners={corners:2}  area={area:.0f}  perimeter={perimeter:.0f}')

show(img)

---
# Part 2 — Feature Detection

**What is a feature?** A small region that is *easy to find again* in another image.

- Flat area ➜ looks the same everywhere ➜ bad feature
- Edge ➜ we can slide along it ➜ okay
- **Corner** ➜ changes in every direction ➜ **best feature** ✅

So most detectors are **corner detectors**.

## 2.1 Harris corner detector
`cv.cornerHarris(gray, blockSize, ksize, k)` gives a "corner score" for each pixel. High score = corner.

In [ ]:
img = cv.imread(DATA + 'chessboard.png')     # any image with corners
gray = np.float32(cv.cvtColor(img, cv.COLOR_BGR2GRAY))

dst = cv.cornerHarris(gray, 2, 3, 0.04)
img[dst > 0.01 * dst.max()] = [0, 0, 255]    # mark strong corners in red

show(img, titles=['Harris corners'])

## 2.2 Shi-Tomasi — `goodFeaturesToTrack`
An improved Harris. We simply ask for **the N best corners**.

In [ ]:
img = cv.imread(DATA + 'chessboard.png')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

corners = cv.goodFeaturesToTrack(gray, maxCorners=50, qualityLevel=0.01, minDistance=10)
for x, y in np.int32(corners).reshape(-1, 2):
    cv.circle(img, (x, y), 5, (0, 255, 0), -1)

show(img, titles=['Shi-Tomasi: best 50 corners'])

## 2.3 ORB — keypoints + descriptors
Harris and Shi-Tomasi only tell us **where** the corners are.

ORB (free & fast alternative to SIFT/SURF) gives us:
- **keypoints**: location, size and angle of each feature
- **descriptors**: a "fingerprint" (vector) for each keypoint ➜ lets us **compare** features between images

It also works when the image is **rotated or scaled**.

In [ ]:
img = cv.imread(DATA + 'messi5.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

orb = cv.ORB_create(nfeatures=300)
kp, des = orb.detectAndCompute(gray, None)
print('Keypoints:', len(kp), '| Descriptor shape:', des.shape)

out = cv.drawKeypoints(img, kp, None, color=(0, 255, 0),
                       flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show(out, titles=['ORB keypoints (circle = size, line = angle)'])

## 2.4 Feature matching
Find an object inside a bigger scene:
1. ORB on both images
2. `BFMatcher` (Brute-Force) compares every descriptor in image 1 with every one in image 2
3. Sort by distance (smaller = more similar) and draw the best matches

In [ ]:
obj   = cv.imread(DATA + 'box.png', cv.IMREAD_GRAYSCALE)           # the object
scene = cv.imread(DATA + 'box_in_scene.png', cv.IMREAD_GRAYSCALE)  # where to look

orb = cv.ORB_create()
kp1, des1 = orb.detectAndCompute(obj, None)
kp2, des2 = orb.detectAndCompute(scene, None)

bf = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=True)   # Hamming distance for ORB
matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)

result = cv.drawMatches(obj, kp1, scene, kp2, matches[:20], None, flags=2)
show(result, titles=['Best 20 matches'])

---
# Part 3 — Object Detection

## 3.1 Template matching
Slide a small image (template) over the big image and measure the similarity at every position.
`minMaxLoc` gives us the best position.

- `TM_SQDIFF` methods ➜ best match = **minimum**
- other methods ➜ best match = **maximum**

⚠️ Works only if the object has the **same size and rotation** as the template.

In [ ]:
img = cv.imread(DATA + 'messi5.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
template = cv.imread(DATA + 'messi_face.jpg', cv.IMREAD_GRAYSCALE)
h, w = template.shape

res = cv.matchTemplate(gray, template, cv.TM_CCOEFF_NORMED)
min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)

top_left = max_loc
cv.rectangle(img, top_left, (top_left[0] + w, top_left[1] + h), (0, 255, 0), 3)
print(f'Match score: {max_val:.2f}')

show(res, img, titles=['Similarity map (bright = similar)', 'Detected'])

**Try it:** compare all 6 methods.

In [ ]:
methods = ['TM_CCOEFF', 'TM_CCOEFF_NORMED', 'TM_CCORR',
           'TM_CCORR_NORMED', 'TM_SQDIFF', 'TM_SQDIFF_NORMED']

plt.figure(figsize=(15, 8))
for i, meth in enumerate(methods):
    method = getattr(cv, meth)
    res = cv.matchTemplate(gray, template, method)
    min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)
    top_left = min_loc if method in [cv.TM_SQDIFF, cv.TM_SQDIFF_NORMED] else max_loc

    out = gray.copy()
    cv.rectangle(out, top_left, (top_left[0] + w, top_left[1] + h), 255, 3)
    plt.subplot(2, 3, i + 1), plt.imshow(out, cmap='gray'), plt.title(meth), plt.axis('off')
plt.show()

## 3.2 Motion detection
Idea: **what changed between two frames = what moved**.

`absdiff` ➜ gray ➜ blur ➜ threshold ➜ dilate (fill holes) ➜ contours ➜ draw boxes around big contours.

In [ ]:
cap = cv.VideoCapture(DATA + 'vtest.avi')
_, frame1 = cap.read()
_, frame2 = cap.read()

while cap.isOpened():
    diff = cv.absdiff(frame1, frame2)
    gray = cv.cvtColor(diff, cv.COLOR_BGR2GRAY)
    blur = cv.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv.threshold(blur, 20, 255, cv.THRESH_BINARY)
    dilated = cv.dilate(thresh, None, iterations=3)
    contours, _ = cv.findContours(dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        if cv.contourArea(cnt) < 700:             # ignore small noise
            continue
        x, y, w, h = cv.boundingRect(cnt)
        cv.rectangle(frame1, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv.imshow('motion detection', frame1)
    cv.imshow('mask', dilated)

    frame1 = frame2
    ret, frame2 = cap.read()
    if not ret or cv.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

## 3.3 Haar cascades — face & eye detection
A **pre-trained** classifier that scans the image with simple black/white patterns (Haar features).
The XML files come with OpenCV (`cv.data.haarcascades`).

`detectMultiScale(gray, scaleFactor, minNeighbors)`
- `scaleFactor=1.1` ➜ shrink the image by 10% each step (to find faces of different sizes)
- `minNeighbors=4` ➜ higher = fewer but more reliable detections

Trick: search for eyes **only inside the face (ROI)** ➜ faster and fewer mistakes.

In [ ]:
face_cascade = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade  = cv.CascadeClassifier(cv.data.haarcascades + 'haarcascade_eye_tree_eyeglasses.xml')

cap = cv.VideoCapture(0)
while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)

    for (x, y, w, h) in face_cascade.detectMultiScale(gray, 1.1, 4):
        cv.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 3)
        roi_gray, roi_color = gray[y:y + h, x:x + w], img[y:y + h, x:x + w]

        for (ex, ey, ew, eh) in eye_cascade.detectMultiScale(roi_gray):
            cv.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 0), 2)

    cv.imshow('face detection', img)
    if cv.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

---
# Summary

| Task | Tool | Good for |
|---|---|---|
| Find corners | Harris / Shi-Tomasi | simple corner points |
| Describe & compare features | ORB + BFMatcher | finding an object even if rotated/scaled |
| Find an exact copy | Template matching | same size & angle only |
| Find moving things | Frame difference + contours | static camera |
| Find faces / eyes | Haar cascade | fast, pre-trained, classic |
| Find objects by color | HSV + `inRange` | colored objects |

**Next step:** deep-learning detectors (YOLO, SSD) — learn the features automatically and detect many classes.